In [3]:
import pandas as pd
import requests
import numpy as np
from pathlib import Path 
from bs4 import BeautifulSoup, Comment
from io import StringIO

pd.set_option("display.max_columns", None)

In [4]:
url = "https://www.basketball-reference.com/leagues/NBA_2026_standings.html"

r = requests.get(url)

In [5]:

soup = BeautifulSoup(r.content, "html.parser")

wrapper = soup.find("div", id="all_expanded_standings")

table_comment = wrapper.find(
    string=lambda text: isinstance(text, Comment)
    and 'id="expanded_standings"' in text
)

table_soup = BeautifulSoup(table_comment, "html.parser")
table = table_soup.find("table", id="expanded_standings")

standings_df = pd.read_html(StringIO(str(table)))[0]

standings_df.columns = [col[1] for col in standings_df.columns.to_flat_index()]

for col in standings_df.columns:
    if col in ["Rk", "Team"]:
        continue 

    standings_df[f"{col}_wins"] = standings_df[col].str.split("-").str[0].astype("int")
    standings_df[f"{col}_losses"] = standings_df[col].str.split("-").str[1].astype("int")
    standings_df = standings_df.drop(columns=[col])

In [6]:
standings_df = standings_df.rename(columns = {
    "Rk" : "rank",
    "Team" : "team",
    "Overall_wins" : "wins",
    "Overall_losses" : "losses",
    "Home_wins" : "home_wins",
    "Home_losses" : "home_losses",
    "Road_wins" : "away_wins",
    "Road_losses" : "away_losses",
    "E_wins" : "east_wins",
    "E_losses" : "east_losses",
    "W_wins" : "west_wins",
    "W_losses" : "west_losses",
    "A_wins" : "atlantic_wins",
    "A_losses" : "atlantic_losses",
    "C_wins" : "central_wins",
    "C_losses" : "central_losses",
    "SE_wins" : "southeast_wins",
    "SE_losses" : "southeast_losses",
    "NW_wins" : "northwest_wins",
    "NW_losses" : "northwest_losses",
    "P_wins" : "pacific_wins",
    "P_losses" : "pacific_losses",
    "SW_wins" : "southwest_wins",
    "SW_losses" : "southwest_losses",
    "Pre_wins" : "pre_allstar_wins",
    "Pre_losses" : "pre_allstar_losses",
    "Post_wins" : "post_allstar_wins",
    "Post_losses" : "post_allstar_losses",
    "≤3_wins" : "within_3pt_wins",
    "≤3_losses" : "within_3pt_losses",
    '≥10_wins' : "outside_10pt_wins",
    '≥10_losses' : "outside_10pt_losses",
    "Oct_wins" : "october_wins",
    "Oct_losses" : "october_losses",
    "Nov_wins" : "november_wins",
    "Nov_losses" : "november_losses",
    "Dec_wins" : "december_wins",
    "Dec_losses" : "december_losses",
    "Jan_wins" : "january_wins",
    "Jan_losses" : "january_losses",
    "Feb_wins" : "february_wins",
    "Feb_losses" : "february_losses",
    "Mar_wins" : "march_wins",
    "Mar_losses" : "march_losses",
    "Apr_wins" : "april_wins",
    "Apr_losses" : "april_losses",
    "May_wins" : "may_wins",
    "May_losses" : "may_losses"
})

In [7]:
east_teams = [
    "Detroit Pistons", "Boston Celtics", "New York Knicks", "Cleveland Cavaliers", "Atlanta Hawks", "Toronto Raptors",
    "Orlando Magic", "Philadelphia 76ers", "Charlotte Hornets", "Miami Heat", "Milwaukee Bucks", "Chicago Bulls", 
    "Brooklyn Nets", "Indiana Pacers", "Washington Wizards"
]

standings_df.loc[standings_df.team.isin(east_teams), "conference"] = "east"
standings_df.loc[~standings_df.team.isin(east_teams), "conference"] = "west"

In [8]:
atlantic_teams = ["Boston Celtics", "New York Knicks", "Brooklyn Nets", "Toronto Raptors", "Philadelphia 76ers"]
central_teams = ["Detroit Pistons", "Milwaukee Bucks", "Chicago Bulls", "Indiana Pacers", "Cleveland Cavaliers"]
southeast_teams = ["Miami Heat", "Orlando Magic", "Charlotte Hornets", "Atlanta Hawks", "Washington Wizards"]

pacific_teams = ["Golden State Warriors", "Los Angeles Lakers", "Los Angeles Clippers", "Sacramento Kings", "Phoenix Suns"]
northwest_teams = ["Portland Trail Blazers", "Denver Nuggets", "Oklahoma City Thunder", "Utah Jazz", "Minnesota Timberwolves"]
southwest_teams = ["Memphis Grizzlies", "Houston Rockets", "Dallas Mavericks", "New Orleans Pelicans", "San Antonio Spurs"]

for division, name in zip(
    [atlantic_teams, central_teams, southeast_teams, pacific_teams, northwest_teams, southwest_teams],
    ["atlantic", "central", "southeast", "pacific", "northwest", "southwest"]
 ):
    standings_df.loc[standings_df.team.isin(division), "division"] = name

In [9]:
standings_df["win_percentage"] = standings_df["wins"] / (standings_df["wins"] + standings_df["losses"])
standings_df["home_win_percentage"] = standings_df["home_wins"] / (standings_df["home_wins"] + standings_df["home_losses"])
standings_df["away_win_percentage"] = standings_df["away_wins"] / (standings_df["away_wins"] + standings_df["away_losses"])

In [10]:
expanded_standings = standings_df.copy()

In [11]:

wrapper = soup.find("div", id="all_team_vs_team")

table_comment = wrapper.find(
    string=lambda text: isinstance(text, Comment)
    and 'id="team_vs_team"' in text
)

table_soup = BeautifulSoup(table_comment, "html.parser")
table = table_soup.find("table", id="team_vs_team")

standings_df = pd.read_html(StringIO(str(table)))[0]

In [12]:
standings_df = standings_df.drop(columns = ["Rk"]).fillna("0-0")

In [13]:
standings_df.columns = ["Team"] + standings_df.Team.tolist()

In [14]:
team_vs_team = standings_df.copy()

In [17]:
def get_conference_rank(expanded_standings, team_vs_team):
    df = expanded_standings.copy() 
    tvt = team_vs_team.copy() 

    conference_rk = 1

    ties_history = []
    conference_rank = {
        "east" : {},
        "west" : {}
    }

    for conference in df.conference.unique():
        conference_df = df[df.conference.eq(conference)].sort_values("win_percentage", ascending=False)

        for rn, row in conference_df.iterrows():
            if row["team"] in conference_rank[conference]:
                continue 

            wp = row["win_percentage"]
            n_ties = len(conference_df[conference_df.win_percentage.eq(wp)]) - 1

            if not n_ties: 
                conference_rank[conference][row["team"]] = conference_rk
                conference_rk += 1
            
            elif n_ties == 1:
                tie_rows = conference_df[conference_df.win_percentage.eq(wp)]
                tie_teams = tie_rows.team.to_list()

                if tie_teams in ties_history:
                    continue 
                
                else:
                    ties_history.append(tie_teams)

                    team0_h2h_wins = int(tvt.loc[tvt.Team.eq(tie_teams[0]), tie_teams[1]].iloc[0].split("-")[0])
                    team1_h2h_wins = int(tvt.loc[tvt.Team.eq(tie_teams[1]), tie_teams[0]].iloc[0].split("-")[0])

                    if team0_h2h_wins > team1_h2h_wins:
                        conference_rank[conference][tie_teams[0]] = conference_rk 
                        conference_rank[conference][tie_teams[1]] = conference_rk + 1
                        conference_rk += 2

                    elif team1_h2h_wins > team0_h2h_wins:
                        conference_rank[conference][tie_teams[1]] = conference_rk 
                        conference_rank[conference][tie_teams[0]] = conference_rk + 1
                        conference_rk += 2

                    else:
                        division_ranks = conference_df.copy() 
                        division_ranks["division_rank"] = conference_df.groupby("division")["win_percentage"].rank(ascending=False, method="dense")

                        division_winners = division_ranks[division_ranks.division_rank.eq(1)].team.tolist()

                        if tie_teams[0] in division_winners and tie_teams[1] not in division_winners:
                            conference_rank[conference][tie_teams[0]] = conference_rk 
                            conference_rank[conference][tie_teams[1]] = conference_rk + 1
                            conference_rk += 2
                        
                        elif tie_teams[1] in division_winners and tie_teams[0] not in division_winners:
                            conference_rank[conference][tie_teams[1]] = conference_rk 
                            conference_rank[conference][tie_teams[0]] = conference_rk + 1
                            conference_rk += 2
                        
                        else:
                            team0_division = tie_rows[tie_rows.team.eq(tie_teams[0])].division.iloc[0]
                            team1_division = tie_rows[tie_rows.team.eq(tie_teams[1])].division.iloc[0]

                            team0_div_wp = int(tie_rows.loc[tie_rows.team.eq(tie_teams[0]), f"{team0_division}_wins"].iloc[0]) / \
                                (
                                    int(tie_rows.loc[tie_rows.team.eq(tie_teams[0]), f"{team0_division}_wins"].iloc[0]) + 
                                    int(tie_rows.loc[tie_rows.team.eq(tie_teams[0]), f"{team0_division}_losses"].iloc[0])
                                )

                            team1_div_wp = int(tie_rows.loc[tie_rows.team.eq(tie_teams[1]), f"{team1_division}_wins"].iloc[0]) / \
                                (
                                    int(tie_rows.loc[tie_rows.team.eq(tie_teams[1]), f"{team1_division}_wins"].iloc[0]) + 
                                    int(tie_rows.loc[tie_rows.team.eq(tie_teams[1]), f"{team1_division}_losses"].iloc[0])
                                )
                            
                            if team0_div_wp > team1_div_wp:
                                conference_rank[conference][tie_teams[0]] = conference_rk 
                                conference_rank[conference][tie_teams[1]] = conference_rk + 1
                                conference_rk += 2

                            elif team1_div_wp > team0_div_wp:
                                conference_rank[conference][tie_teams[1]] = conference_rk 
                                conference_rank[conference][tie_teams[0]] = conference_rk + 1
                                conference_rk += 2

        if len(conference_rank[conference]) < len(conference_df):
            raise ValueError("Need to revisit the function code to add more tiebreakers")
        for tm, rank in conference_rank[conference].items():
            df.loc[df.team.eq(tm), "conference_rank"] = rank

        conference_rk = 1      
                    
    return df

In [18]:
expanded_standings = get_conference_rank(expanded_standings, team_vs_team)

,rank,team,wins,losses,home_wins,home_losses,away_wins,away_losses,east_wins,east_losses,west_wins,west_losses,atlantic_wins,atlantic_losses,central_wins,central_losses,southeast_wins,southeast_losses,northwest_wins,northwest_losses,pacific_wins,pacific_losses,southwest_wins,southwest_losses,pre_allstar_wins,pre_allstar_losses,post_allstar_wins,post_allstar_losses,within_3pt_wins,within_3pt_losses,outside_10pt_wins,outside_10pt_losses,october_wins,october_losses,november_wins,november_losses,december_wins,december_losses,january_wins,january_losses,february_wins,february_losses,march_wins,march_losses,april_wins,april_losses,conference,division,win_percentage,home_win_percentage,away_win_percentage,conference_rank
0,1,Oklahoma City Thunder,64,18,34,8,30,10,23,7,41,11,8,2,7,3,8,2,12,4,17,2,12,5,42,14,22,4,5,6,43,9,6,0,14,1,9,4,9,6,8,4,14,1,4,2,west,northwest,0.780488,0.809524,0.750000,1.0
1,2,San Antonio Spurs,62,20,32,8,30,12,26,4,36,16,9,1,8,2,9,1,10,8,13,5,13,3,38,16,24,4,8,5,43,8,5,0,8,6,11,3,8,7,11,0,14,2,5,2,west,southwest,0.756098,0.800000,0.714286,2.0
2,3,Detroit Pistons,60,22,32,9,28,13,39,13,21,9,15,3,12,4,12,6,8,2,7,3,6,4,40,13,20,9,10,6,34,8,3,2,13,2,9,4,10,4,9,2,11,7,5,1,east,central,0.731707,0.780488,0.682927,1.0
3,4,Boston Celtics,56,26,30,11,26,15,36,16,20,10,10,6,12,6,14,4,3,7,10,0,7,3,35,19,21,7,5,7,38,10,3,3,8,6,9,3,10,6,9,2,11,5,6,1,east,atlantic,0.682927,0.731707,0.634146,2.0
4,5,Denver Nuggets,54,28,28,13,26,15,18,12,36,16,7,3,5,5,6,4,11,5,11,6,14,5,35,20,19,8,9,11,28,9,3,2,11,3,9,5,10,6,4,7,11,5,6,0,west,northwest,0.658537,0.682927,0.634146,3.0
5,6,Los Angeles Lakers,53,29,28,13,25,16,20,10,33,19,7,3,6,4,7,3,10,7,10,7,13,5,33,21,20,8,8,3,28,21,4,2,11,2,5,7,9,7,6,6,15,2,3,3,west,pacific,0.646341,0.682927,0.609756,4.0
6,7,New York Knicks,53,29,30,10,23,19,35,17,18,12,14,3,10,7,11,7,7,3,4,6,7,3,35,20,18,9,9,4,32,19,2,3,11,3,10,4,7,8,8,4,10,6,5,1,east,atlantic,0.646341,0.750000,0.547619,3.0
7,8,Cleveland Cavaliers,52,30,27,14,25,16,33,19,19,11,8,8,11,5,14,6,5,5,7,3,7,3,34,21,18,9,2,5,27,17,3,3,9,6,7,7,10,5,8,3,10,5,5,1,east,central,0.634146,0.658537,0.609756,4.0
8,9,Houston Rockets,52,30,30,11,22,19,23,7,29,23,7,3,8,2,8,2,8,10,11,7,10,6,33,20,19,10,5,9,27,11,2,2,11,2,7,6,10,7,7,5,9,7,6,1,west,southwest,0.634146,0.731707,0.536585,5.0
9,10,Minnesota Timberwolves,49,33,26,15,23,18,18,12,31,21,6,4,6,4,6,4,9,7,9,9,13,5,34,22,15,11,6,4,28,18,2,3,10,5,9,5,10,6,6,4,9,6,3,4,west,northwest,0.597561,0.634146,0.560976,6.0
